In [ ]:
import os
path_to_frontend = '..'
os.chdir(path_to_frontend)
import q100viz.session as session
import matplotlib.pyplot as plt
import pandas
import datetime

def GAMA_time_to_datetime(input):
    dt_object = int(datetime.datetime.strptime(input[7:-11], '%Y-%m-%d').year)
    return(dt_object)
def grams_to_tons(val):
    return val / 1000000

## load source data:

In [ ]:

# source data:
df_emissions = pandas.read_csv("/home/dunland/github/qScope/data/outputs/output_20221105_13-02-44/emissions/CO2_emissions_neighborhood.csv")
df_emissions['current_date'] = df_emissions['current_date'].apply(GAMA_time_to_datetime)
df_emissions['emissions_neighborhood_total'] = df_emissions['emissions_neighborhood_total'].apply(grams_to_tons)

df_connections = pandas.read_csv("/home/dunland/github/qScope/data/outputs/output_20221105_13-02-44/connections/connections_export.csv")
df_connections['current_date'] = df_connections['current_date'].apply(GAMA_time_to_datetime)
df_connections['value'] = df_connections['value'] * len(session.buildings.df) / 100

# reference data:
df_emissions_compare = pandas.read_csv("/home/dunland/github/qScope/data/precomputed/simulation_defaults/emissions/CO2_emissions_neighborhood.csv")
df_emissions_compare['current_date'] = df_emissions_compare['current_date'].apply(GAMA_time_to_datetime)
df_emissions_compare['emissions_neighborhood_total'] = df_emissions_compare['emissions_neighborhood_total'].apply(grams_to_tons)

df_connections_compare = pandas.read_csv("/home/dunland/github/qScope/data/precomputed/no_buildings_selected/connections/connections_export.csv")
df_connections_compare['current_date'] = df_connections_compare['current_date'].apply(GAMA_time_to_datetime)


## line plot:
pands-dataframe-based, cannot combine multiple y-axes into one legend

In [ ]:

# plot:
fig = plt.figure(figsize=(12,8))
ax0 = plt.axes()  # all graphs shall be in the same figures
plt.title("Emissionen und W?rmenetzanschl?sse", fontsize='x-large')

##################### left y-axis: ####################
df_emissions.plot(
    kind='line',
    x='current_date',
    y=['emissions_neighborhood_total'],
    ax=ax0,
    label=["j?hrliche Emissionen"],
    color='black'
)

df_emissions_compare.plot(
    kind='line',
    x='current_date',
    y=['emissions_neighborhood_total'],
    ax=ax0,
    color='lightgray',
    label=["j?hrliche Emissionen (unver?ndert)"]
)
plt.gca().set_ylabel("$CO_{2}$-?quivalente (t)", fontsize='x-large')
plt.yticks(fontsize='x-large')
plt.xticks(fontsize='x-large')
ax0.legend(fontsize='x-large')

#################### right y-axis: ####################
df_connections.plot(
    kind='line',
    x='current_date',
    y=['value'],
    secondary_y=True,
    mark_right=False,
    ax=ax0,
    label=['Anschl?sse'],
    color='#3f9236'
)
plt.gca().set_ylabel('Anzahl W?rmenetzanschl?sse', fontsize='x-large')
plt.gca().set_yticks(range(0, len(session.buildings.df), 10),fontsize='x-large')
plt.xticks(fontsize='x-large')

df_connections_compare.plot(
    kind='line',
    x='current_date',
    y=['value'],
    secondary_y=True,
    mark_right=False,
    ax=ax0,
    color='#73876d',
    label=['Anschl?sse (unver?ndert)'],
)

plt.yticks(fontsize='x-large')
ax0.set_xlabel('Jahr', fontsize='x-large')
ax0.set_xticks(range(2020,2050,5), fontsize='x-large')
# ax0.legend(fontsize='x-large')


## sharing the legend properly:
by using ax.plot instead of df.plot, the labels can be combined (with adjusted fontsize) into one legend

In [ ]:
# plot:
fig = plt.figure(figsize=(12,8))
ax0 = plt.axes()  # all graphs shall be in the same figures
plt.title("Quartiersemissionen und W?rmenetzanschl?sse", fontsize='x-large')

##################### left y-axis: ####################
line1, = ax0.plot(
    df_emissions['current_date'],
    df_emissions['emissions_neighborhood_total'],
    color='black'
)

line2, = ax0.plot(
    df_emissions_compare['current_date'],
    df_emissions_compare['emissions_neighborhood_total'],
    color='lightgray'
)
plt.gca().set_ylabel("$CO_{2}$-?quivalente (t)", fontsize='x-large')
plt.yticks(fontsize='x-large')
plt.xticks(fontsize='x-large')

#################### right y-axis: ####################
ax1 = ax0.twinx()
line3, = ax1.plot(
    df_connections['current_date'],
    df_connections['value'],
    color='#00a84e'
)
plt.gca().set_ylabel('Anzahl W?rmenetzanschl?sse', fontsize='x-large')
plt.gca().set_yticks(range(0, len(session.buildings.df), 10),fontsize='x-large')
plt.xticks(fontsize='x-large')

line4, = ax1.plot(
    df_connections_compare['current_date'],
    df_connections_compare['value'],
    color='gray'
)

plt.yticks(fontsize='x-large')
ax0.set_xlabel('Jahr', fontsize='x-large')
ax0.set_xticks(range(2020,2050,5), fontsize='x-large')
plt.legend([line1, line2, line3, line4], ["j?hrliche Emissionen", "j?hrliche Emissionen (unver?ndert)",'Anschl?sse','Anschl?sse (unver?ndert)'], loc='upper left', fontsize='x-large')


## bar plots:

In [ ]:
# plot:
fig = plt.figure(figsize=(12,8))
ax0 = plt.axes()  # all graphs shall be in the same figures
plt.title("Quartiersemissionen und W?rmenetzanschl?sse", fontsize='x-large')

##################### left y-axis: ####################
barplot_green = ax0.bar(
    df_connections.iloc[::365, :]['current_date'],
    df_connections.iloc[::365, :]['value'],
    color='#00a84e',
    alpha=0.5
)
plt.gca().set_ylabel('Anzahl W?rmenetzanschl?sse', fontsize='x-large')
plt.gca().set_yticks(range(0, len(session.buildings.df), 10),fontsize='x-large')


barplot_gray = ax0.bar(
    df_connections_compare['current_date'],
    df_connections_compare['value'],
    color='#9f9f9f50', 
    alpha=0.5
)
ax0.set_xlabel('Jahr', fontsize='x-large')
plt.yticks(fontsize='x-large')
plt.xticks(fontsize='x-large')

#################### right y-axis: ####################
ax1 = ax0.twinx()
line_black, = ax1.plot(
    df_emissions['current_date'],
    df_emissions['emissions_neighborhood_total'],
    color='black',
    linewidth=3
)

line_gray, = ax1.plot(
    df_emissions_compare['current_date'],
    df_emissions_compare['emissions_neighborhood_total'],
    color='lightgray',
    linewidth=3
)
plt.gca().set_ylabel("$CO_{2}$-?quivalente (t)", fontsize='x-large')
ax1.set_ylim(bottom=0)

plt.yticks(fontsize='x-large')
plt.legend([barplot_green, barplot_gray, line_black, line_gray], ['Anschl?sse','Anschl?sse (unver?ndert)', "j?hrliche Emissionen", "j?hrliche Emissionen (unver?ndert)"], loc='lower center', fontsize='x-large')

